# 07 · 수식으로 정의하고 실제 모델로 검증하기

05는 기존 A/B/C/D 실험, 06은 기존 결과 검증·비교, **07은 수식 정의 → 관측 대조 → 새 2단계 구조의 실험**을 담당한다.
1~9절은 기존 로그의 검산과 명시된 가정의 함수값이다. 10~14절은 실제 Jev/Laya로 개발 설정을 평가하고,
정책 고정 후 이전에 평가하지 않은 문항에서 A/B와 비교한다.

기본 실행은 **유료 호출 없이** 완료한 실험을 읽는다. 10절의 `RUN_PARALLEL_EXPERIMENT=True`일 때 새 실험을 실행한다.
문항 수·그룹 크기·선택 수는 `config/parallel_routing.json`에서 바꾼다.
현재 corpus는 정답 근거를 강제로 포함한 2,000문단 축소본이며, 전체 KILT나 생산 서비스 성능을 주장하지 않는다.

15~18절은 **g=20, K=5를 고정한 대표 설명 길이 실험**이다. 기존 결과와 설정을 보존하고 개발 40문항만 사용한다.
`config/representation_length.json`에서 설명 길이와 호출 한도를 설정한다. 기본 `RUN_LENGTH_STUDY=False`는
마지막 완료 결과만 검증·표시하며, True일 때 설명 준비 및 Jev 검색의 유료 API를 호출한다.


## 1. 무엇을 수학적으로 구할 것인가

| 기호 | 정의·단위 |
|---|---|
| $D=\{d_1,\ldots,d_N\}$ | 전처리된 원문 passage 집합, $N$은 passage 수 |
| $q$, $h$ | 질문, 임베딩 차원 |
| $\mathcal G_q=\{G_{q1},\ldots\}$ | 정답 annotation별 **완전한 근거 ID 집합**. 대안 annotation은 하나만 완성해도 됨 |
| $L$, $k$ | dense 후보 수, 최종 LLM에 전달할 근거 수 |
| $H_q$, $E_q$, $Y_q$ | 후보 완전 근거 포함, 전달 근거 완전 포함, 정규화 Exact Match 성공의 0/1 지표 |
| $T$, $U$, $C$ | 실제 경과 시간(초), 처리/청구 토큰 수, 비용(USD). 모델별 토큰은 별도 단위 |
| $g$, $M=\lceil N/g\rceil$, $r$ | 새 구조의 묶음 최대 크기, 묶음 수, 선택 묶음 수. $r$은 최종 근거 수 $k$와 다름 |
| $a$, $b$ | 한 묶음 설명, 한 원문 판단의 가정된 평균 입력 토큰. 실측으로 정하기 전에는 모형 상수 |
| $\alpha$, $\beta$ | 필요한 묶음을 여는 비율(TPR), 불필요한 묶음을 여는 비율(FPR) |
| $\theta$ | 묶음 규칙·요약 길이·임계값·탐색 한도·배치 설정 등의 정책 |

필요한 결과는 $R_{\rm evidence}(\theta)$, $EM(\theta)$, $F1(\theta)$, $T(\theta)$, $U(\theta)$,
$C_{\rm API}(\theta)$와 준비·갱신 비용이다. API 청구액에 로컬 GPU·사람의 작업 비용이 없다고 해서 그 비용을 0으로 두지는 않는다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json
import math
import sys

# Windows의 PyArrow DLL 충돌을 피하려고 torch 의존 모듈보다 먼저 import한다.
import pandas as pd
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, ticker
from IPython.display import display, Markdown

ROOT = Path.cwd() if (Path.cwd() / 'rag_baselines').exists() else Path.cwd().parent
if not (ROOT / 'rag_baselines').is_dir():
    raise FileNotFoundError('프로젝트 루트 또는 notebook 폴더에서 실행하세요.')
sys.path.insert(0, str(ROOT))
from rag_baselines.comparison import load_comparison, resolve_comparison_run
from rag_baselines.data import file_sha256, read_json, write_json
from rag_baselines.evaluation import notebook_source_sha256
from rag_baselines.generation import usage_cost
from rag_baselines.math_analysis import evidence_decomposition, group_input_bound

SYSTEM_RUN = None  # 최근 완료 런. 특정 results 하위 런 또는 staged_* 경로로 변경 가능.
RUN_DIR, STUDY_DIR = resolve_comparison_run(ROOT, SYSTEM_RUN)
print(f'원본: {RUN_DIR}')

font_path = Path('C:/Windows/Fonts/malgun.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams.update({'font.size': 11, 'axes.unicode_minus': False,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'text.color': '#263746', 'axes.labelcolor': '#263746'})
COLORS = {'A': '#416D95', 'B': '#BE7D32', 'C': '#537D59', 'D': '#8565A8'}

### 완료 런과 실제 조건 확인

06과 같은 검증기를 사용한다. 원문·질문·라벨 해시, 완전 근거 판정, EM/F1, 실제 프롬프트,
usage와 가격, A/B 동일 후보, 선언된 문항과 반복 수를 재검증한다. 런에 C/D가 있다면 그 기록도 함께 검사한다.
계산에는 **해당 런의 설정 스냅샷**을 사용한다. 현재 설정 파일로 과거 결과를 재해석하지 않는다.

In [ ]:
manifest, rows, source_hashes = load_comparison(ROOT, RUN_DIR)
df = pd.DataFrame(rows)
ab = df[df['method'].isin(['A', 'B'])].copy()
assert set(ab['method']) == {'A', 'B'}
config = manifest['config']
# 아래 A/B 수식이 정의하는 모델·토큰화 규약인지 먼저 확인한다.
if not (config['retriever']['pooling'] == 'cls'
        and config['retriever']['similarity'] == 'dot'
        and config['retriever']['max_length'] == 256
        and config['reranker']['max_length'] == 256
        and config['reranker']['truncation'] == 'only_second'):
    raise ValueError('이 07의 A/B 수식은 CLS·dot product·256토큰·only_second 규약을 대상으로 합니다.')
N = manifest['index']['signature']['count']
L, evidence_k = config['retrieve_k'], config['evidence_k']
OUT = ROOT / 'results' / f"math_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{uuid4().hex[:8]}"
OUT.mkdir(parents=True, exist_ok=False)

conditions = pd.DataFrame([
    {'항목': '문서·질문', '기록': f"{N:,} passages / {ab.question_id.nunique()} 질문 / 반복 {manifest.get('repeats', 1)}"},
    {'항목': '분할', '기록': manifest['split']},
    {'항목': '후보·근거 수', '기록': f'L={L}, k={evidence_k}'},
    {'항목': 'BERGEN commit', '기록': config['bergen_revision']},
    {'항목': 'A retriever revision', '기록': config['retriever']['revision']},
    {'항목': 'B reranker revision', '기록': config['reranker']['revision']},
    {'항목': '공통 생성기', '기록': manifest['generation']['model']},
    {'항목': '측정 실행 시각(UTC)', '기록': manifest['created_utc']},
])
display(conditions)

code_match = []
for name in ['models.py', 'evaluation.py', 'generation.py']:
    relative = Path('rag_baselines') / name
    recorded = manifest.get('code_sha256', {}).get(str(relative))
    current = file_sha256(ROOT / relative)
    code_match.append({'file': str(relative), 'recorded_sha256': recorded,
                       'current_sha256': current, 'same_source': recorded == current if recorded else None})
display(pd.DataFrame(code_match)[['file', 'same_source']])
if any(row['same_source'] is not True for row in code_match):
    display(Markdown('**주의:** 일부 현재 코드와 런의 소스 해시가 다릅니다. 아래 관측치 재계산과 당시 구현 확인을 구분하세요.'))
print(f'계산 결과 저장: {OUT}')

## 2. A/B의 실제 검색 수식

모델 파라미터 $\phi$를 고정하고, 토큰화·최대 256토큰 처리를 $\tau_{256}$으로 쓴다.
실제 문서 입력은 준비된 `content`이며, CLS 출력을 사용한다.

$$v_i=f_\phi(\tau_{256}(d_i)),\qquad u=f_\phi(\tau_{256}(q)),\qquad s_i=u^\top v_i.$$

**A는 정규화 없는 dot product로 전체 $N$개를 exact search**한다. cosine/ANN/BM25 구현이 아니다.
실제 저장 임베딩·행렬곱 dtype과 버전은 manifest에 남아 있으므로 이 식은 실수 연산의 추상화다.

$$\mathcal C_L(q)=\operatorname{TopL}_{d_i\in D}s_i,\qquad
S_A(q)=\operatorname{Topk}_{d_i\in\mathcal C_L(q)}s_i.$$

B는 **같은 후보**에서 질문·원문 쌍을 cross-encoder에 넣어 raw logit $z_i$를 얻는다.
질문은 보존하고 문서만 잘라 쌍 전체를 최대 256토큰으로 처리한다. $z_i$를 보정된 확률로 해석하지 않는다.

$$z_i=r_\psi(q,d_i),\qquad S_B(q)=\operatorname{Topk}_{d_i\in\mathcal C_L(q)}z_i,
\qquad \widehat a_X=G(q,S_X(q)),\ X\in\{A,B\}.$$

A는 모든 점수를 계산한 뒤 `torch.argsort(..., stable=True)`로 **전체 점수 배열을 정렬**한다.
동점은 corpus 순서로 해소한다. 반환·생성에는 검색 모델이 읽은 잘린 문장이 아니라 **원문**을 사용한다.
구현 근거: [models.py](../rag_baselines/models.py), [generation.py](../rag_baselines/generation.py).

### 입력 손실도 숫자로 확인

$U_{\rm original}$, $U_{\rm processed}$, $U_{\rm padded}$를 구분하고
$U_{\rm dropped}=U_{\rm original}-U_{\rm processed}$를 확인한다.
아래는 실제 런의 토큰화 감사 기록 합계다. 패딩은 실제 내용 길이나 API 청구량과 같지 않다.

In [ ]:
stage_rows, token_rows = [], []
index_tokens = manifest['index'].get('document_tokens', {})
if index_tokens:
    token_rows.append({'input': '공통 문서 인덱스(준비 1회)', **index_tokens})
with (RUN_DIR / 'system.jsonl').open(encoding='utf-8') as stream:
    for line in stream:
        if not line.strip():
            continue
        record = json.loads(line)
        retrieved, generated = record['retrieval'], record['generation']
        method = retrieved['method']
        if method not in ['A', 'B']:
            continue
        timings = retrieved['seconds']
        generation_seconds = sum(generated['seconds'][name] for name in ['prompt_prepare', 'api_request'])
        named_retrieval = sum(timings[name] for name in ['query_encode', 'vector_search', 'rerank', 'assemble'])
        stage_rows.append({'method': method, 'task': retrieved['task'],
            'question_id': retrieved['question_id'], 'repeat': retrieved['repeat'],
            **{name: timings[name] for name in ['query_encode', 'vector_search', 'rerank', 'assemble']},
            'retrieval_remainder': timings['retrieval_total'] - named_retrieval,
            'retrieval_total': timings['retrieval_total'], 'generation': generation_seconds,
            'online_remainder': record['online_total_seconds'] - timings['retrieval_total'] - generation_seconds,
            'online_total': record['online_total_seconds']})
        for name, audit in retrieved.get('model_tokens', {}).items():
            if name == 'reranker_pairs' and method == 'A':
                continue
            token_rows.append({'input': f'{method} / {name}', **audit})
stages = pd.DataFrame(stage_rows)
token_columns = ['nonpad', 'original_nonpad', 'padded', 'truncated_sequences', 'dropped_tokens']
tokens = pd.DataFrame(token_rows).groupby('input')[token_columns].sum(min_count=1).reset_index()
assert (tokens.original_nonpad - tokens.nonpad == tokens.dropped_tokens).all()
assert len(stages) == len(ab)
assert not stages.duplicated(['method', 'question_id', 'repeat']).any()
display(tokens)

## 3. 정확도를 분해하는 식

선택 집합 $S$의 완전 근거 포함 여부를 다음처럼 정의한다.

$$R_q(S)=\mathbf 1\{\exists G\in\mathcal G_q:\ G\subseteq S\},\quad
H_q=R_q(\mathcal C_L(q)),\quad E_q=R_q(S_X(q)),\quad
Y_q=\mathbf1\{\operatorname{norm}(\widehat a_X)\in\operatorname{norm}(\text{정답 별칭})\}.$$

HotpotQA의 필수 근거 두 개 중 하나만 있으면 완전 근거 성공이 아니다. 여러 유효 annotation 중 하나를 완성하면 성공이다.
A/B는 $S_X\subseteq\mathcal C_L$이므로 $E\Rightarrow H$이고,
**독립성 가정 없이** 다음 항등식이 성립한다.

$$P(E)=P(H)P(E\mid H).$$

답변 EM은 전체 확률 법칙으로 분해한다.

$$P(Y)=P(E)P(Y\mid E)+[1-P(E)]P(Y\mid\neg E).$$

$P(Y\mid\neg E)$를 미리 0으로 두지 않는다. 정답 근거 ID가 누락돼도 모델 지식·다른 유효 자료·우연히 정답을 낼 수 있다.
조건 사건이 0개면 조건부 비율은 **미정(None/NaN)**으로 남기고, 그 사건의 가중 기여는 0이다.
이것은 표본의 회계적 분해이며 인과 설명이나 새 질문의 정확도 예측식이 아니다.

In [ ]:
quality_rows = []
for (method, task), group in ab.groupby(['method', 'task']):
    observations = [(int(h), int(e), int(y)) for h, e, y in
        group[['complete_candidate_hit', 'complete_evidence_hit', 'answer_em']].itertuples(index=False, name=None)]
    quality_rows.append({'method': method, 'task': task, **evidence_decomposition(observations)})
for method, group in ab.groupby('method'):
    observations = [(int(h), int(e), int(y)) for h, e, y in
        group[['complete_candidate_hit', 'complete_evidence_hit', 'answer_em']].itertuples(index=False, name=None)]
    quality_rows.append({'method': method, 'task': 'all', **evidence_decomposition(observations)})
quality = pd.DataFrame(quality_rows)
for row in quality.to_dict('records'):
    candidate_product = (row['candidate_rate'] * row['evidence_given_candidate']
                         if row['candidate_hits'] else 0.0)
    assert math.isclose(candidate_product, row['evidence_rate'], abs_tol=1e-12)
    assert math.isclose(row['covered_em_contribution'] + row['uncovered_em_contribution'],
                        row['answer_em'], abs_tol=1e-12)
display(quality[['method', 'task', 'observations', 'candidate_hits', 'evidence_hits',
    'covered_correct', 'uncovered_correct', 'evidence_given_candidate',
    'correct_given_evidence', 'correct_without_evidence', 'answer_em']].round(4))
for row in quality[quality.task == 'all'].to_dict('records'):
    display(Markdown(f"**{row['method']}**: 후보 {row['candidate_hits']}/{row['observations']} → "
        f"완전 근거 {row['evidence_hits']}/{row['observations']} → "
        f"EM {row['covered_correct'] + row['uncovered_correct']}/{row['observations']} = **{row['answer_em']:.1%}**. "
        f"근거 미포함 상태의 정답 {row['uncovered_correct']}개."))

**해석의 한계:** 완전 근거 ID 포함은 내용상 충분성과 같지 않고, EM 실패는 생성기만의 잘못이라는 뜻이 아니다.
라벨 정합성·정답 별칭·근거 내 모호함을 질문별로 확인해야 한다. 원래 EM/F1을 보존하고
[answer_audit.json](../config/answer_audit.json)의 검토 대상도 별도로 본다. 반복이 있으면 위 분모는 질문×반복 관측 수이며 독립 표본 수가 아니다.

## 4. 시간 식: 무엇이 얼마나 걸렸는가

$$T_A=T_{\rm query}+T_{\rm search}+T_{\rm assemble}+T_{\rm ret\_rest}+T_{\rm gen}+T_{\rm online\_rest},$$
$$T_B=T_{\rm query}+T_{\rm search}+T_{\rm rerank}+T_{\rm assemble}+T_{\rm ret\_rest}+T_{\rm gen}+T_{\rm online\_rest}.$$

$T_{\rm gen}$은 프롬프트 준비+실제 API 요청이다. 두 rest 항은 기록된 전체 시간과 세부 합계의 차이이며 원인별로 세분됐다고 주장하지 않는다.
아래 계산에는 로그 저장 오버헤드가 포함된다. GPU 동기화·캐시·동시 부하·배치 설정도 실제 시간에 영향을 준다.

계산량의 추상 모델은

$$W_A=C_f(q)+O(Nh)+S(N),\qquad
W_B=W_A+\sum_{i\in\mathcal C_L}C_r(q,d_i).$$

$S(N)$은 **전체 argsort 구현의 작업량**이다. CPU 비교 정렬의 $O(N\log N)$을 GPU 구현에 확인 없이 대입하지 않는다.
Cross-encoder batch가 $B_r$이면 forward batch 수는 $\lceil L/B_r\rceil$이다. batch 시간은 길이·패딩·하드웨어에 따라 달라진다.
$C_f,C_r$에는 학습된 의미 계산이 이미 들어 있으므로 A/B도 AI를 쓰는 알고리즘이다.

In [ ]:
time_columns = ['query_encode', 'vector_search', 'rerank', 'assemble',
                'retrieval_remainder', 'generation', 'online_remainder']
assert np.allclose(stages[time_columns].sum(axis=1), stages.online_total, atol=1e-9, rtol=0)
assert (stages[['retrieval_remainder', 'online_remainder']] >= -1e-6).all().all()
time_summary = stages.groupby('method')[time_columns + ['retrieval_total', 'online_total']].mean()
display(time_summary.round(6))
print(f"B의 후보 {L}개 / batch {config['reranker']['batch_size']} → "
      f"질문당 forward batch {math.ceil(L / config['reranker']['batch_size'])}회")

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), layout='constrained')
overall = quality[quality.task == 'all'].set_index('method').loc[['A', 'B']]
metric_names = [('candidate_rate', '후보에 완전 근거'), ('evidence_rate', '전달한 완전 근거'), ('answer_em', '답변 EM')]
x = np.arange(len(metric_names))
for offset, method in [(-.18, 'A'), (.18, 'B')]:
    values = [overall.loc[method, name] for name, _ in metric_names]
    bars = axes[0].bar(x + offset, values, width=.34, color=COLORS[method], label=method)
    axes[0].bar_label(bars, labels=[f'{value:.0%}' for value in values], padding=3)
axes[0].set_xticks(x, [label for _, label in metric_names])
axes[0].set_ylim(0, 1.15)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(1))
axes[0].set_ylabel('질문×반복 관측 비율')
axes[0].set_title(f'A/B 품질 · 방법당 {int(overall.observations.iloc[0])} 관측')
axes[0].legend(frameon=False, loc='upper right')
phase_values = pd.DataFrame({
    '질문 임베딩': time_summary.query_encode,
    '벡터 검색': time_summary.vector_search,
    '재정렬': time_summary.rerank,
    '생성': time_summary.generation,
    '조립·나머지': time_summary.assemble + time_summary.retrieval_remainder + time_summary.online_remainder,
})
phase_colors = ['#416D95', '#86A8C5', '#BE7D32', '#C6CED5', '#6D7D89']
bottom = np.zeros(2)
for column, color in zip(phase_values.columns, phase_colors):
    values = phase_values.loc[['A', 'B'], column].to_numpy()
    axes[1].bar(['A', 'B'], values, bottom=bottom, color=color, label=column)
    bottom += values
axes[1].set_ylim(0, max(bottom) * 1.25)
for index, value in enumerate(bottom):
    axes[1].text(index, value + max(bottom)*.025, f'{value:.3f}초', ha='center', fontweight='bold')
axes[1].set_ylabel('문항당 평균 경과 시간 (초)')
axes[1].set_title('기록된 시간의 합계 · 로딩·준비 제외')
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
for ax in axes:
    ax.grid(axis='y', color='#E3E8EC', linewidth=.7)
    ax.set_axisbelow(True)
fig.savefig(OUT / 'ab_accounting.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. 토큰·비용 식: 청구와 계산량을 구분

실제 usage에서 총 입력 $I$, 캐시 읽기 $I_c$, 캐시 쓰기 $I_w$, 출력 $O$를 읽는다.
각 가격 $p$의 단위는 USD/백만 토큰이다. 당시 가격 스냅샷으로 추정 사용료를 계산한다.

$$C_G={ (I-I_c-I_w)p_i+I_cp_c+I_wp_w+Op_o\over10^6},\qquad
C_{\rm API}=C_{\rm retrieval\ API}+C_G.$$

Jev는 **실제 온라인 호출 usage 합계**로 판단 청구량을 계산한다. 같은 응답의 캐시 재사용을 새 API 청구로 더하지 않는다.
Laya의 처리 토큰은 로컬 처리량이며 API 청구 토큰이 아니다. 다른 토크나이저의 토큰을 더해 FLOP나 속도라고 해석하지 않는다.
키·인증 헤더·`.env`는 이 분석에 필요 없다.

$$C_{\rm amortized/query}={C_{\rm initial\ prep}+\sum C_{\rm update}\over Q_{\rm served}}
+C_{\rm online\ API}+C_{\rm local}+C_{\rm human}.$$

준비·로컬·사람 비용을 모르면 총비용은 **미측정**이다. 준비 인덱스의 최초 build 시간과 재사용 load/검증 시간을 구별한다.
참조: [usage_cost](../rag_baselines/generation.py), [원시 로그](../research/raw-logs.md).

### API 비용의 손익분기 조건

판단기 입력 가격이 $p_J>0$, 출력 가격이 0이고 A 검색의 API 청구액이 0인 조건에서

$$C_J+C_G^{\rm new}\le C_G^A
\quad\Longrightarrow\quad
U_J\le{10^6(C_G^A-C_G^{\rm new})\over p_J}.$$

생성 비용 절감분이 음수면 입력 예산 상한도 음수이므로 비음수 usage를 만족하는 해가 없다.
아래는 **기존 C 생성 비용을 그대로 유지한다는 조건**의 예산 대조다. 새 구조의 생성 비용·품질을 예측하는 식은 아니다.
0토큰 생성 비용을 가정한 절대 예산도 함께 계산하지만 달성 가능한 품질을 보장하지 않는다.
준비·로컬·사람 비용을 포함한 총비용 손익분기는 별도로 구한다.


In [ ]:
cost_rows = []
with (RUN_DIR / 'system.jsonl').open(encoding='utf-8') as stream:
    for line in stream:
        if not line.strip():
            continue
        record = json.loads(line)
        generation = record['generation']
        recalculated = usage_cost(generation['api_usage'], generation['prices'])
        recorded_cost = record['generation_api_cost_usd']
        assert (recalculated is None and recorded_cost is None) or (
            recalculated is not None and recorded_cost is not None
            and math.isclose(recalculated, recorded_cost, abs_tol=1e-12))
        details = (generation.get('api_usage') or {}).get('input_tokens_details') or {}
        cost_rows.append({'method': record['retrieval']['method'],
            'question_id': record['retrieval']['question_id'], 'repeat': record['retrieval']['repeat'],
            'cached_input_tokens': details.get('cached_tokens', 0) if isinstance(generation.get('api_usage'), dict) else None,
            'cache_write_tokens': details.get('cache_write_tokens', 0) if isinstance(generation.get('api_usage'), dict) else None,
            'recomputed_generation_usd': recalculated})
cost_audit = pd.DataFrame(cost_rows)
api_costs = df.groupby('method')[['retrieval_api_cost_usd', 'generation_api_cost_usd', 'online_api_cost_usd']].agg(
    lambda values: values.mean() if values.notna().all() else np.nan)
display(api_costs.round(9))
display(cost_audit.groupby('method')[['cached_input_tokens', 'cache_write_tokens']].agg(
    lambda values: values.sum() if values.notna().all() else np.nan))
display(Markdown(f"공통 인덱스 기록: 최초 build **{manifest['index']['build_seconds']:.4f}초**, "
    f"이번 준비 **{manifest['index']['prepare_seconds']:.4f}초**, cache_hit={manifest['index']['cache_hit']}. "
    '이 값은 전체 문서 준비·초기 다운로드 비용이 아닙니다.'))

breakeven = pd.DataFrame()
if 'C' in api_costs.index:
    jev_prices = manifest['c']['config'].get('prices') or {}
    price = jev_prices.get('input_usd_per_million')
    if price is not None and price > 0 and jev_prices.get('output_usd_per_million') == 0:
        a_generation = api_costs.loc['A', 'generation_api_cost_usd']
        c_generation = api_costs.loc['C', 'generation_api_cost_usd']
        saving = a_generation - c_generation
        breakeven = pd.DataFrame([{
            'A_생성비_USD': a_generation, 'C_생성비_USD': c_generation,
            '생성_절감_USD': saving, 'Jev_입력가격_USD_per_M': price,
            '같은_C_생성비의_Jev_입력예산_tokens': saving * 1e6 / price if pd.notna(saving) else np.nan,
            '생성비_0일때_Jev_입력예산_tokens': a_generation * 1e6 / price,
            '기존_C_Jev_실측_tokens': df[df.method == 'C'].jev_input_tokens.mean(),
        }])
        display(breakeven.T.rename(columns={0: '문항당 평균 · 기존 비용 조건'}).round(9))


## 6. AI를 포함한 알고리즘의 작업량과 순차 의존성

의미 판단기는 oracle $J$로 표시하되, 호출·입력·결과 오류를 무료 연산으로 두지 않는다.
현재 C/D의 탐색은 방문 노드·원문·근거 충분성 판단의 합계로 기록할 수 있다.

$$U_J(q)=\sum_{j\in\text{실제 처리 호출}}U_j(q),\qquad
W_J(q)=\sum_{j\in\text{판단 작업}}w_j(q).$$

청구 토큰 $U_J$와 실제 계산량 $W_J$는 같은 양이 아니다. 원문 길이·패딩·typed question 수·재시도·캐시 상태를 함께 기록한다.
같은 라운드의 판단은 병렬화할 수 있어도, 다음 라운드에 필요한 정보가 아직 없으면 순차 의존성이 남는다.

고정된 장치의 처리량을 $\rho$ 연산/초, 의존 경로의 compute 시간을 $S$초라 하면

$$T_{\rm compute}\ge\max\left({W_{\rm operations}\over\rho},S\right).$$

총시간에는 네트워크·대기·코드·저장도 들어간다. **논리적 단계 2개는 입력량이나 실행 시간이 $O(1)$이라는 증거가 아니다.**
한 호출에 담을 수 있는 작업 수·입력 한도·배치 정책을 실제로 확인해야 한다.
현재 C/D를 새로운 두 단계 구조의 실측으로 취급하지 않는다.

In [ ]:
decision_columns = ['decision_model_calls', 'jev_input_tokens', 'laya_input_tokens',
                    'retrieval_seconds', 'online_seconds']
decisions = df[df.method.isin(['C', 'D'])]
if not decisions.empty:
    display(decisions.groupby('method')[decision_columns].mean().round(3))
else:
    display(Markdown('선택한 런에 C/D가 없어 기존 판단 경로의 실측 표는 생략합니다.'))

## 7. 새 구조: 묶음 요약 선택 → 원문 판단

원문을 서로 겹치지 않는 묶음 $D_1,\ldots,D_M$으로 준비하고, 요약 $\sigma_j$와 원문 참조를 저장한다.
이 설계에서 요약 작성·적재는 오프라인 작업이다. 요약 자체로 원문 모든 사실이 보존된다고 가정하지 않는다.

$$\pi_j=J(q,\sigma_j),\quad
\mathcal B(q)=\operatorname{Topr}_j\pi_j\ \text{또는}\ \{j:\pi_j\ge\tau\},\quad
\mathcal C^{\rm new}(q)=\bigcup_{j\in\mathcal B(q)}D_j,$$
$$z_i^{\rm new}=J(q,d_i),\quad S_{\rm new}(q)=\operatorname{Select}(\{d_i,z_i^{\rm new}\}_{d_i\in\mathcal C^{\rm new}},k).$$

분기에서는 **질문에 필요한 근거 일부가 있을 수 있는가**를 판단한다. 전체 질문을 혼자 해결해야 한다고 묻지 않는다.
Noul/score/choice의 구체적 출력과 선택 규칙은 개발 실험에서 고정할 대상이다.
`Topr`는 읽기량을 제한하지만 필요한 여러 묶음을 모두 남긴다는 보장이 없다.
임계값 방식은 복수 묶음을 열 수 있지만 불필요한 묶음도 열릴 수 있다.

새 후보의 완전 근거 포함을 $H_{\rm new}$라 하면

$$P(E_{\rm new})=P(H_{\rm new})P(E_{\rm new}\mid H_{\rm new}).$$

요약 누락·상위 묶음 선택 실패는 첫 항, 원문 선택 실패는 둘째 항에 반영된다. 이 식에도 독립성은 필요 없다.
원문을 읽은 뒤에야 다음 단서가 생기는 멀티홉 질문은 **추가 라운드**가 필요할 수 있다.

### 7.1 고정 선택 수의 입력량 모형

모든 요약을 한 번 판단하고, 선택한 $r$개 묶음의 원문을 한 번씩 판단한다고 가정한다.
묶음 최대 크기가 $g$라면 아래 식은 입력량의 **용량 상한 모형**이다.

$$\widehat U(g,r)=a\left\lceil{N\over g}\right\rceil+b\min(N,rg).$$

각 입력 길이가 $a,b$라는 상수인 모형 안에서의 상한이다. 표본 평균을 넣은 식은 실제 usage의 상한이 아니다.
지시문을 상수에 배분했으며 요청 헤더·공유 state 반복·재시도 등은 제외한다.
실제 usage와 비교할 때 그 항들을 별도로 더해야 한다.
$1\le g\le N/r$에서 천장 함수를 무시한 연속 완화는

$$U_0(g,r)={aN\over g}+brg,\quad
{\partial U_0\over\partial g}=-{aN\over g^2}+br,\quad
{\partial^2 U_0\over\partial g^2}={2aN\over g^3}>0.$$

따라서 내부 해는

$$g^*=\sqrt{{aN\over br}},\qquad U_0(g^*,r)=2\sqrt{abrN}.$$

최적점이 범위 밖이면 경계로 제한한다. 정수·불균등 묶음·문맥 길이·정확도 제약은 따로 확인한다.
이 $O(\sqrt N)$ 주장은 **$a,b,r$이 고정되고 $g$를 $N$에 맞춰 바꿀 때의 입력량 모형**에만 해당한다.
$g$를 50으로 고정하면 $aN/50+50br$은 $N$에 선형이다.
요약 길이 $a$가 묶음 크기에 따라 증가하거나 필요한 묶음 수가 늘면 같은 결과가 유지된다고 할 수 없다.

In [ ]:
# 수식의 함수값 확인용 가정. 현재 Jev/Laya의 실측 추정치나 새 구조의 성능 예측이 아니다.
THEORY = {'N': 2000, 'r': 2, 'a': 40.0, 'b': 150.0,
          'needed_groups': 2, 'alpha': .95, 'beta': .02}
if any(type(THEORY[key]) is not int or THEORY[key] < 1 for key in ['N', 'r', 'needed_groups']):
    raise ValueError('N, r, needed_groups는 양의 정수여야 합니다.')
if any(not isinstance(THEORY[key], (int, float)) or not math.isfinite(THEORY[key])
       or not 0 <= THEORY[key] <= 1 for key in ['alpha', 'beta']):
    raise ValueError('TPR/FPR은 0~1의 유한한 값이어야 합니다.')
# 첫 경계에서도 토큰 상수·선택 수 조건을 검증한다.
group_input_bound(THEORY['N'], 1, THEORY['r'], THEORY['a'], THEORY['b'])
if THEORY['needed_groups'] > THEORY['N']:
    raise ValueError('필요한 묶음 수는 전체 원문 수를 넘을 수 없습니다.')
n, r, a, b = (THEORY[key] for key in ['N', 'r', 'a', 'b'])
g_star = math.sqrt(a*n / (b*r))
g_star_feasible = min(max(g_star, 1.0), n/r)
grid = pd.DataFrame([{'g': g, **group_input_bound(n, g, r, a, b)}
                     for g in range(1, n//r + 1)])
best = grid.loc[grid.total_tokens_upper_bound.idxmin()]
continuous_min = a*n/g_star_feasible + b*r*g_star_feasible
assert (grid.total_tokens_upper_bound >= continuous_min - 1e-9).all()
display(pd.DataFrame([{'연속 내부 최적 g': g_star, '범위 제한 g': g_star_feasible,
    '연속 최소 입력량 모형': continuous_min, '정수 상한 모형 최소 g': int(best.g),
    '정수 상한 모형 최소 토큰': best.total_tokens_upper_bound}]).round(3))
display(Markdown('이 최솟값은 **가정한 입력량 식의 최솟값**입니다. 정답 근거 회수율·실행 속도·실제 청구량은 아직 검증하지 않았습니다.'))

### 7.2 오탐률이 남기는 비용

현재 묶음 중 필요한 묶음이 $m$, 필요/불필요 묶음의 평균 선택 비율이 각각 $\alpha,\beta$라면,
선택한 묶음 수 $K$의 기대값은 **독립성 없이** 선형성으로

$$\mathbb E[K]=m\alpha+(M-m)\beta.$$

모든 필요한 묶음을 여는 확률까지 $\alpha^m$으로 쓰려면 별도의 독립성 가정이 필요하다.
정답 근거의 대안 경로가 있으면 필요한 묶음의 정의부터 다시 정해야 한다.

균등 묶음($M=N/g$)을 가정하고 $a,b,m,\alpha,\beta$를 상수로 놓은 완화 모형은

$$\mathbb E[U]\approx{aN\over g}+bg\left[m\alpha+\left({N\over g}-m\right)\beta\right]
={aN\over g}+bm(\alpha-\beta)g+b\beta N.$$

따라서 고정된 양의 오탐률 $\beta$가 남으면 **$b\beta N$의 선형 항**도 남는다.
$m$은 실제로 질문·묶음 방식·$g$에 따라 달라질 수 있고 $\alpha,\beta$도 달라질 수 있다.
둘 다 상수로 놓은 위 식을 실험 결과로 주장하지 않는다.

In [ ]:
# 균등 묶음의 연속 완화이며, 모든 값은 THEORY의 가정으로 계산한다.
m, alpha, beta = (THEORY[key] for key in ['needed_groups', 'alpha', 'beta'])
plot_grid = grid[grid.g <= min(n//r, n//m)].copy()
plot_grid['expected_selected_groups'] = m*alpha + (n/plot_grid.g - m)*beta
plot_grid['relaxed_threshold_tokens'] = a*n/plot_grid.g + b*plot_grid.g*plot_grid.expected_selected_groups
plot_grid['fixed_g50_tokens'] = a*n/50 + b*r*50

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), layout='constrained')
for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel('묶음 최대 크기 g (passages, 로그 축)')
    ax.set_ylabel('가정한 입력 토큰 수')
    ax.grid(color='#E3E8EC', linewidth=.7)
axes[0].plot(plot_grid.g, plot_grid.total_tokens_upper_bound, color='#416D95', label='정수 용량 상한 모형')
axes[0].plot(plot_grid.g, a*n/plot_grid.g + b*r*plot_grid.g,
             color='#263746', linestyle='--', label='연속 완화')
axes[0].scatter([best.g], [best.total_tokens_upper_bound], color='#416D95', s=45, zorder=3)
axes[0].annotate(f"g={int(best.g)}, {best.total_tokens_upper_bound:,.0f}토큰",
    (best.g, best.total_tokens_upper_bound), xytext=(10, 18), textcoords='offset points')
axes[0].set_title(f'고정 r={r} · 입력량 식의 최솟값')
axes[0].legend(frameon=False, loc='upper right')
axes[1].plot(plot_grid.g, plot_grid.relaxed_threshold_tokens, color='#BE7D32', label='임계값 선택의 완화 모형')
axes[1].axhline(b*beta*n, color='#263746', linestyle='--', label=f'오탐 선형 항 bβN={b*beta*n:,.0f}')
axes[1].set_title(f'가정: TPR={alpha:.0%}, FPR={beta:.0%}, m={m}')
axes[1].legend(frameon=False, loc='upper right')
# 데이터를 그린 뒤 실제 범위를 지정한다. 0~1의 초기 축을 고정하면 곡선이 사라진다.
axes[0].set_ylim(0, plot_grid.total_tokens_upper_bound.max() * 1.1)
axes[1].set_ylim(0, max(plot_grid.relaxed_threshold_tokens.max(), b*beta*n) * 1.1)
fig.suptitle(f'관측·성능 예측 아님 · N={n:,}, a={a:g}, b={b:g}의 수식 함수값', fontsize=12)
fig.savefig(OUT / 'two_stage_input_model.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. 검증할 연구 가설과 실험 순서

입력량 최솟값을 찾는 것과 품질·비용·시간을 함께 만족시키는 것은 다른 문제다. 예를 들어 정책은

$$\min_\theta\mathbb E[T_\theta]\quad\text{subject to}\quad
R_{\rm evidence}(\theta)\ge R_{\min},\quad EM(\theta)\ge A_{\min},\quad
\mathbb E[C_{\rm API,\theta}]\le C_{\max}.$$

$R_{\min},A_{\min},C_{\max}$와 비교 기준·허용 열화 폭은 **새 평가 전에 정한다**.
또는 동일 품질 구간에서 시간·비용의 Pareto 경계를 비교한다. 임의 가중치로 셋을 섞어 승리를 만들지 않는다.

| 구할 양 | 정의·검증 방식 | 현재 상태 |
|---|---|---|
| $P(H),P(E\mid H),P(Y\mid E),P(Y\mid\neg E)$ | 원래 근거 라벨·정답 별칭으로 계산 | 기존 A/B 로그에서 검산 |
| $T_{\rm query},T_{\rm search},T_{\rm rerank},T_{\rm gen}$ | 실측 경과 시간·로그 범위 확인 | 기존 A/B 로그에서 검산 |
| $a(g),b,\alpha(g),\beta(g),m(q,g)$ | 개발 자료에서 묶음/원문 입력과 사람 검증 라벨로 추정 | 새 구조에서는 미측정 |
| $K(q),$ 누락 근거, 추가 라운드 | 선택 묶음·원문·실제 근거 위치·단서별 trace | 새 구조 구현 후 기록 |
| 호출 수·처리량·패딩·잘림·재시도 | 논리 단계와 실제 API 요청/GPU forward를 모두 기록 | 새 구조에서는 미측정 |
| 준비·갱신·로컬·사람 비용 | 같은 공통 전처리, 별도 적재·요약 비용, 질의 수별 상각 | 완전한 총비용은 미측정 |

1. **개발:** 여러 $N,g,r,\tau$ 조건에서 실제 입력·판단 오류·지연을 측정한다. 단일 자료와 여러 자료 질문을 구분한다.
2. **적합:** 개발에서만 $a,b,\alpha,\beta$ 및 시간 모형 계수를 추정한다. 토큰을 FLOP/초로 자동 변환하지 않는다.
3. **고정:** grouping·요약 생성 모델·입력·선택 정책·batch·실험 순서·quality target을 기록하고 동결한다.
4. **새 평가:** 아직 정책 선택에 사용하지 않은 질문·규모에서 **예측값과 실측값의 잔차**를 본다.
   $e_T(q)=T_{\rm measured}(q)-\widehat T(q)$, $e_C(q)=C_{\rm measured}(q)-\widehat C(q)$와 품질 제약 위반을 저장한다.
5. **공정한 비교:** 같은 corpus·질문·원문 반환·생성기·예산을 쓰고 품질 평가는 후보 재사용, 지연 평가는 실제 검색으로 분리한다.
   같은 배포 조건·동시성·준비 상각 범위를 선언한다. A/B도 최적화할 기회를 같게 준다.

현재 평가 결과에 맞춰 새 구조를 튜닝한 뒤 같은 문항으로 검증 완료를 선언하지 않는다.
숫자 계수가 확인되지 않은 속도/비용 역전은 가설로 남긴다.

## 9. 저장과 근거

질문별 A/B 관측·확률 분해·시간 항·입력 잘림·비용 검산·수식 함수값과 출처 해시를 새 폴더에 저장한다.
원래 study·raw 로그·05/06은 수정하지 않는다. 그림은 현재 선택한 원본 또는 명시된 `THEORY` 식에서만 만든다.

- 로컬 정의: [baseline-contract.md](../research/baseline-contract.md), [evaluation.py](../rag_baselines/evaluation.py),
  [math_analysis.py](../rag_baselines/math_analysis.py).
- 기준 프레임워크: [고정 BERGEN](https://github.com/naver/bergen/tree/eb8ec046eb6cf08be95b035b2d3bcc4b241ff212).
  현재 구현은 Windows adapter이며 공식 전체 benchmark 재현을 주장하지 않는다.
- typed 판단·병렬 요청의 기능 설명: [TypeSafe 모델](https://docs.typesafe.ai/models),
  [Fan-out](https://docs.typesafe.ai/patterns/fan-out). 현재 런의 usage/가격과 API 기능 문서는 구분한다.
- 묶음 요약과 계층 검색의 관련 연구: [RAPTOR](https://arxiv.org/abs/2401.18059).
  위 묶음/판단 방식이 같은 알고리즘이라는 뜻은 아니며, 새롭다는 결론도 아직 내리지 않는다.

이후 새 알고리즘도 **변수·가정 → 수식 → 개발 측정 → 정책 고정 → 새 평가의 검산** 형식으로 이어간다.

In [ ]:
exports = {
    'ab_observations.csv': ab,
    'quality_decomposition.csv': quality,
    'ab_stage_timings.csv': stages,
    'ab_mean_timings.csv': time_summary.reset_index(),
    'input_truncation_audit.csv': tokens,
    'generation_cost_audit.csv': cost_audit,
    'api_breakeven_conditions.csv': breakeven,
    'api_mean_costs.csv': api_costs.reset_index(),
    'theoretical_group_input_bound.csv': grid,
    'theoretical_threshold_relaxation.csv': plot_grid,
    'source_code_match.csv': pd.DataFrame(code_match),
}
for name, table in exports.items():
    table.to_csv(OUT / name, index=False, encoding='utf-8-sig')
inputs = dict(source_hashes)
if (RUN_DIR / 'completed.json').exists():
    inputs[str((RUN_DIR / 'completed.json').relative_to(ROOT))] = file_sha256(RUN_DIR / 'completed.json')
for name in ['math_analysis.py', 'models.py', 'evaluation.py', 'generation.py', 'comparison.py']:
    path = ROOT / 'rag_baselines' / name
    inputs[str(path.relative_to(ROOT))] = file_sha256(path)
# 반복 읽기 사이에 원본이 바뀌지 않았는지 확인한다.
for name, digest in source_hashes.items():
    assert file_sha256(ROOT / name) == digest, f'분석 중 원본 변경: {name}'
write_json(OUT / 'analysis_manifest.json', {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'analysis': 'mathematical_framework',
    'scope': 'Sections 1-9 only: historical accounting and hypothetical formulas; live experiment has separate receipts', 'system_run': str(RUN_DIR),
    'notebook': 'notebook/07_mathematical_framework.ipynb',
    'notebook_source_sha256': notebook_source_sha256(ROOT / 'notebook/07_mathematical_framework.ipynb'),
    'source_sha256': inputs, 'methods_validated': manifest.get('methods', ['A', 'B', 'C']),
    'ab_observations': len(ab), 'ab_unique_questions': ab.question_id.nunique(),
    'split': manifest['split'], 'theory_parameters': THEORY,
    'theory_scope': 'hypothetical formula values; no inference, quality guarantee or latency prediction',
    'new_policy_validation': 'not executed', 'new_api_requests': 0,
    'exports_sha256': {p.name: file_sha256(p) for p in sorted(OUT.iterdir()) if p.is_file()},
})
print(f'검산 완료 · {len(ab)} A/B 관측 · 새 API 호출 0회')
print(OUT)

## 10. 실제 2단계 라우팅의 조건

기존 Jev 분류 경로·제목 순서로 모든 원문을 중복 없이 묶는다. 묶음별 **짧은 주제 설명＋원문 개체명＋원문 인용 Q&A**는
질문·정답 없이 사전에 생성하고 원문 위치를 유지한다. 압축으로 빠지는 정보가 있으며 완전한 탐색을 보장하지 않는다.
기존 분류의 준비 비용과 새 설명 생성 비용은 별도로 저장한다.

온라인 흐름은 **모든 묶음 Noul → 상위 r개 묶음의 원문 Score → 원문 상위 k개 → 공통 GPT-6 Luna**다.
선택한 r개 묶음을 모두 평가하므로 추가 중단·복구·완결성 호출은 없다. 두 논리 단계와 실제 요청/배치 수를 구분한다.
Jev HTTP 요청은 최대 `http_workers`개를 함께 실행하며, Laya는 기존 pinned 모델을 한 번 로드해 배치로 재사용한다.

개발에서는 r 최댓값까지 실제 판단하고 작은 r의 부분집합은 **품질만 재생**한다. 재생 값의 지연·사용량·요금은 미측정이다.
B의 개발 근거 확보율을 충족하는 설정 중 입력량 proxy가 작은 것을 선택한다. 충족하지 못하면 근거 확보율을 우선하며
`target_met=False`를 남긴다. 이는 실제 요금·시간 최적값을 증명하는 정책이 아니다.
정책과 모델을 고정한 다음 과거 평가 ID를 제외한 새 문항으로 A/B/C-parallel/D-parallel을 직접 실행한다.

In [ ]:
import os
from rag_baselines.parallel_experiment import run_parallel_experiment, load_parallel_report
from rag_baselines.data import read_jsonl

RUN_PARALLEL_EXPERIMENT = False  # True면 새 유료 실험. False면 마지막 완료 결과만 분석.
# 에이전트의 일회 실행용; 일반 노트북 실행에서는 설정할 필요 없음.
RUN_PARALLEL_EXPERIMENT = RUN_PARALLEL_EXPERIMENT or os.environ.get('DECISION_LLM_RUN_PARALLEL') == '1'
PARALLEL_SETTINGS = read_json(ROOT / 'config/parallel_routing.json')
display(pd.DataFrame([PARALLEL_SETTINGS]).T.rename(columns={0: '설정'}))

if RUN_PARALLEL_EXPERIMENT:
    from collections import deque
    from IPython.display import clear_output
    progress_lines = deque(maxlen=12)
    def experiment_progress(message):
        progress_lines.append(message)
        clear_output(wait=True)
        print('실제 모델/API 실행 중 · 자세한 진행 기록은 results/raw 및 partial JSONL에 보존')
        print('\n'.join(progress_lines))
    PARALLEL_DIR = run_parallel_experiment(ROOT, PARALLEL_SETTINGS, progress=experiment_progress)
else:
    pointer = ROOT / 'results/current_parallel_run.txt'
    PARALLEL_DIR = ROOT / pointer.read_text(encoding='utf-8').strip() if pointer.exists() else None
if PARALLEL_DIR is not None:
    dev_routing, heldout, frozen_routing = load_parallel_report(PARALLEL_DIR)
    PARALLEL_SETTINGS = frozen_routing['settings']  # 읽는 실험의 고정 설정으로 표시/계산.
    PARALLEL_ANALYSIS = PARALLEL_DIR / 'analysis'
    PARALLEL_ANALYSIS.mkdir(exist_ok=True)
    print('검증한 실험:', PARALLEL_DIR)
else:
    display(Markdown('**아직 실제 실험 결과가 없습니다.** 실행 셀을 켜기 전에는 아래 수치·그림을 만들지 않습니다.'))

## 11. 회수율을 가정하지 않고 측정하기

$H_q$: 유효한 정답 annotation 하나에 필요한 **모든 묶음**이 선택됨.
$E_q$: 반환한 원문이 그 annotation의 모든 근거를 포함함.

$$\widehat R_{\rm macro}=\frac{\sum H_q}{Q},\quad
\widehat R_{\rm micro\mid macro}=\frac{\sum E_qH_q}{\sum H_q},\quad
\widehat R_{\rm evidence}=\frac{\sum E_q}{Q}.$$

분모가 0인 조건부 확률은 미정이다. 묶음 TPR/FPR은 유효 annotation의 원문 근거가 있는 묶음을 양성으로 표시하고
전체 질문의 TP/양성 수, FP/음성 수로 합산한다. 복수 annotation의 완결성은 별도로 확인한다.
이 측정값은 묶음/질문/모델별 결과이며 그룹 크기에 반비례하는 보편 법칙으로 간주하지 않는다.

In [ ]:
if PARALLEL_DIR is not None:
    display(dev_routing.round(4))
    display(pd.DataFrame(frozen_routing['selected']).T)
    for provider, chosen in frozen_routing['selected'].items():
        if not chosen['target_met']:
            display(Markdown(f"**{provider}: 개발 B의 완전 근거 목표를 충족한 설정이 없습니다.** 최선의 관측 설정을 탐색적 평가에 사용합니다."))
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8), layout='constrained')
    provider_colors = {'jev': COLORS['C'], 'laya': COLORS['D']}
    for (provider, g), part in dev_routing.groupby(['provider', 'group_size']):
        part = part.sort_values('selection_k')
        label = f'{provider} · g={g}'
        style = '-' if g == min(PARALLEL_SETTINGS['group_sizes']) else '--'
        axes[0].plot(part.selection_k, part.complete_evidence_hit, style, color=provider_colors[provider], marker='o', label=label)
        axes[1].plot(part.selection_k, part.input_proxy_tokens, style, color=provider_colors[provider], marker='o', label=label)
    target = next(iter(frozen_routing['selected'].values()))['development_target']
    axes[0].axhline(target, color='#263746', linestyle=':', label=f'B 개발 근거 확보율 {target:.0%}')
    axes[0].set_ylim(0, 1.08)
    axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(1))
    axes[0].set_ylabel('필수 근거 전체 확보율')
    axes[0].set_title('개발 · 실제 판단을 재생한 품질')
    axes[1].set_ylim(bottom=0)
    axes[1].set_ylabel('묶음 설명＋후보 원문 text tokens (proxy)')
    axes[1].set_title('설정 선택용 proxy · 청구량/실측 지연 아님')
    for ax in axes:
        ax.set_xlabel('선택 묶음 수 r')
        ax.set_xticks(PARALLEL_SETTINGS['selection_budgets'])
        ax.grid(color='#E3E8EC', linewidth=.7)
        ax.legend(frameon=False, fontsize=9)
    fig.suptitle(f"개발 {dev_routing.questions.iloc[0]}문항 · corpus {PARALLEL_SETTINGS['corpus_size']:,}문단", fontsize=12)
    fig.savefig(PARALLEL_ANALYSIS / 'parallel_development.png', dpi=160, bbox_inches='tight')
    plt.show()

## 12. 토큰·배치·시간 모형을 실제 기록과 대조하기

개발 로그에서 그룹 판단당 $\hat a=U_{\rm macro}/M$, 원문 판단당 $\hat b=U_{\rm micro}/n_{\rm candidates}$를 측정한다.
고정한 설정의 평가 입력 예측 $\widehat U=\hat aM+\hat b rg$를 **새 평가의 실제 입력량**과 비교한다.
서로 다른 모델의 토큰을 합쳐 계산량으로 간주하지 않는다. 지시문·질문·길이 분포·패킹 차이로 예측 오차가 생길 수 있다.

$$T_{\rm retrieval}=T_{\rm macro,wall}+T_{\rm micro,wall}+T_{\rm prepare/assemble/log}.$$

논리 단계는 2개지만 실제 HTTP 호출 수와 Laya의 예정 forward 배치 수는 별도다.
Laya 시간에는 입력 감사/토큰화가 포함되고 순수 추론 시간도 따로 기록된다. GPU peak allocated는 해당 프로세스의
가중치/기존 인덱스를 포함하며 기기 전체 메모리나 모델만의 추가 메모리가 아니다. 입력 clipping은 SDK 호출 전에 거부한다.
N=2,000의 한 실행으로 $O(1)$ 지연이나 $O(\sqrt N)$ 비용을 검증했다고 주장하지 않는다.

In [ ]:
if PARALLEL_DIR is not None:
    profile_rows = []
    for row in read_jsonl(PARALLEL_DIR / 'development_judgments.jsonl'):
        judged = row['judgments']
        counts = {'macro': len(judged['macro_answers']), 'micro': len(judged['micro_answers'])}
        for stage in ['macro', 'micro']:
            calls = [c for c in judged['calls'] if c['stage'] == stage]
            tokens_stage = sum(c['response']['usage']['input_tokens'] for c in calls)
            profile_rows.append({'provider': row['provider'], 'g': row['group_size'], 'question_id': row['question_id'],
                'stage': stage, 'judgments': counts[stage], 'input_tokens': tokens_stage,
                'tokens_per_judgment': tokens_stage / counts[stage], 'wall_seconds': judged['seconds'][stage],
                'physical_requests': len(calls) if row['provider'] == 'jev' else 0,
                'scheduled_local_batches': sum(math.ceil(len(c['request']['questions'])/c['request']['batch_size']) for c in calls) if row['provider'] == 'laya' else None})
    profiles = pd.DataFrame(profile_rows)
    measured_parameters = profiles.groupby(['provider', 'g', 'stage'])[['tokens_per_judgment', 'input_tokens', 'wall_seconds', 'judgments', 'physical_requests']].mean()
    display(measured_parameters.round(3))
    live_records = read_jsonl(PARALLEL_DIR / 'system.jsonl')
    input_checks = []
    for provider, chosen in frozen_routing['selected'].items():
        g, r = chosen['group_size'], chosen['selection_k']
        a_measured = measured_parameters.loc[(provider, g, 'macro'), 'tokens_per_judgment']
        b_measured = measured_parameters.loc[(provider, g, 'micro'), 'tokens_per_judgment']
        predicted = a_measured * math.ceil(PARALLEL_SETTINGS['corpus_size']/g) + b_measured*r*g
        method = 'C-parallel' if provider == 'jev' else 'D-parallel'
        matches = [row['retrieval'] for row in live_records if row['retrieval']['method'] == method]
        observed = np.mean([row['model_tokens'][provider]['nonpad'] for row in matches])
        input_checks.append({'provider': provider, 'g': g, 'r': r, 'a_dev': a_measured, 'b_dev': b_measured,
            'predicted_eval_input': predicted, 'observed_eval_input': observed,
            'fixed_ab_stationary_group_size': math.sqrt(a_measured*PARALLEL_SETTINGS['corpus_size']/(b_measured*r)),
            'relative_error': (predicted-observed)/observed,
            'mean_physical_http_requests': np.mean([row['trace']['physical_http_requests'] for row in matches]),
            'mean_scheduled_local_batches': np.mean([row['trace']['scheduled_local_batches'] for row in matches]) if provider == 'laya' else None,
            'mean_local_inference_seconds': np.mean([row['trace']['local_inference_seconds'] for row in matches]) if provider == 'laya' else None,
            'max_process_peak_allocated_MiB': max(row['trace']['local_peak_allocated_bytes'] for row in matches)/1024**2 if provider == 'laya' else None,
            'truncated_retrievals': sum(row['trace']['input_truncated'] for row in matches)})
    measured_inputs = pd.DataFrame(input_checks)
    display(measured_inputs.round(3))
    profiles.to_csv(PARALLEL_ANALYSIS / 'stage_profiles.csv', index=False)
    measured_inputs.to_csv(PARALLEL_ANALYSIS / 'measured_input_model.csv', index=False)

## 13. 고정한 설정의 새 문항 비교

A/B와 새 두 방법 모두 같은 원문·질문·5개 근거 예산·생성기·평가기·원시 로그를 사용한다. A/B 모델·리비전은 기존과 같다.
방법 순서는 질문마다 무작위화하며 실제 검색을 매번 실행한다. 정책을 고른 개발 품질 재생을 지연 측정으로 재사용하지 않는다.
**API 비용은 로컬 GPU·사전 준비·사람의 작업 비용을 제외한 청구 추정값**이다.
EM의 막대 오차선은 질문별 이항 결과의 95% Wilson 구간이며, 방법 간 차이의 유의성 검정은 아니다.
한 번의 작은 평가 표본에서 일반적인 우위를 확정하지 않는다.

In [ ]:
if PARALLEL_DIR is not None:
    display(heldout[['method', 'questions', 'complete_evidence_hit', 'answer_em', 'answer_f1', 'retrieval_seconds', 'online_seconds', 'llm_input_tokens', 'decision_input_tokens', 'online_api_cost_usd']].round(6))
    table = heldout.set_index('method').loc[['A', 'B', 'C-parallel', 'D-parallel']]
    names = ['A · 기본 RAG', 'B · 재정렬 RAG', 'C · Jev 2단계', 'D · Laya 2단계']
    colors = [COLORS[m] for m in ['A', 'B', 'C', 'D']]
    positions = np.arange(4)
    fig, axes = plt.subplots(2, 2, figsize=(13.6, 9), layout='constrained')
    estimates, low, high = [], [], []
    for method, row in table.iterrows():
        p, count, z = row.answer_em, row.questions, 1.96
        center = (p+z*z/(2*count))/(1+z*z/count)
        half = z*math.sqrt(p*(1-p)/count+z*z/(4*count*count))/(1+z*z/count)
        estimates.append(p); low.append(max(0, p-(center-half))); high.append(max(0, center+half-p))
    bars = axes[0,0].bar(positions, estimates, color=colors, yerr=[low, high], capsize=4)
    axes[0,0].bar_label(bars, labels=[f'{v:.0%}' for v in estimates], padding=5)
    axes[0,0].set_ylim(0, 1.15)
    axes[0,0].yaxis.set_major_formatter(ticker.PercentFormatter(1))
    axes[0,0].set_ylabel('정답률 EM · 95% Wilson 구간')
    axes[0,0].set_title('실제 생성 답변 정확도')
    for ax, column, title, unit, fmt in [
        (axes[0,1], 'online_seconds', '실제 전체 응답 시간', '초 / 질문', lambda v:f'{v:.2f}초'),
        (axes[1,0], 'llm_input_tokens', '공통 생성기 입력', '청구 입력 토큰 / 질문', lambda v:f'{v:,.0f}'),
        (axes[1,1], 'online_api_cost_usd', '온라인 API 비용 · 준비·GPU 비용 제외', 'USD / 질문', lambda v:f'${v:.6f}')]:
        values = table[column].to_numpy()
        bars = ax.bar(positions, values, color=colors)
        ax.bar_label(bars, labels=[fmt(v) for v in values], padding=4, fontsize=10)
        ax.set_ylim(0, values.max()*1.22)
        ax.set_title(title); ax.set_ylabel(unit)
    for ax in axes.flat:
        ax.set_xticks(positions, names, fontsize=10)
        ax.grid(axis='y', color='#E3E8EC', linewidth=.7)
        ax.set_axisbelow(True)
    fig.suptitle(f"이전 평가 ID 제외 · {int(table.questions.iloc[0])}문항 · {PARALLEL_SETTINGS['corpus_size']:,}문단 · 반복 1회", fontsize=13)
    fig.savefig(PARALLEL_ANALYSIS / 'parallel_heldout.png', dpi=160, bbox_inches='tight')
    plt.show()
    write_json(PARALLEL_ANALYSIS / 'analysis_manifest.json', {
        'experiment': str(PARALLEL_DIR), 'system_sha256': file_sha256(PARALLEL_DIR/'system.jsonl'),
        'frozen_policy_sha256': file_sha256(PARALLEL_DIR/'frozen_policy.json'),
        'notebook_source_sha256': notebook_source_sha256(ROOT/'notebook/07_mathematical_framework.ipynb'),
        'scope': 'Actual bounded development and held-out measurements; no asymptotic or accuracy guarantee.',
        'exports_sha256': {p.name:file_sha256(p) for p in PARALLEL_ANALYSIS.iterdir() if p.is_file() and p.name!='analysis_manifest.json'}})
    print('실측·수식 대조·비교 그림:', PARALLEL_ANALYSIS)

## 14. 이번 측정의 해석

아래 해석은 완료한 실험에서 자동 계산한다. 고정 설정의 작은 평가에서 관측한 결과이며, 이전 트리 실험과 질문이 다르므로 개선율을 직접 비교하지 않는다.
단가 a,b를 고정한 비용식의 정지점은 참고 계산이고 품질·실제 시간 최적값의 증명이 아니다.


In [ ]:
if PARALLEL_DIR is not None:
    a_row, b_row = table.loc['A'], table.loc['B']
    c_row, d_row = table.loc['C-parallel'], table.loc['D-parallel']
    c_micro = c_row.complete_evidence_hit / c_row.complete_candidate_hit if c_row.complete_candidate_hit else float('nan')
    display(Markdown(
        f"**새 평가 {int(c_row.questions)}문항의 관측:** A/B/Jev 2단계/Laya 2단계 EM은 "
        f"{a_row.answer_em:.0%}/{b_row.answer_em:.0%}/{c_row.answer_em:.0%}/{d_row.answer_em:.0%}. "
        f"Jev의 필수 근거 확보율은 {c_row.complete_evidence_hit:.0%}, B는 {b_row.complete_evidence_hit:.0%}다.\n\n"
        f"Jev는 질문당 판단 입력 {c_row.decision_input_tokens:,.0f}토큰, 실제 HTTP 요청 "
        f"{measured_inputs.loc[measured_inputs.provider == 'jev', 'mean_physical_http_requests'].iloc[0]:.1f}회, "
        f"전체 {c_row.online_seconds:.2f}초, API 비용 ${c_row.online_api_cost_usd:.6f}를 사용했다. "
        f"B 대비 전체 시간 {c_row.online_seconds/b_row.online_seconds:.2f}배, API 비용 "
        f"{c_row.online_api_cost_usd/b_row.online_api_cost_usd:.1f}배다. 준비·GPU 금전 비용은 제외한다.\n\n"
        f"Jev 후보 안에 필수 근거가 모두 있는 비율은 {c_row.complete_candidate_hit:.0%}, "
        f"그 조건에서 최종 근거를 모두 전달한 비율은 {c_micro:.0%}다. "
        "이 실행의 주된 근거 누락은 묶음 선택에서 발생했다. 이는 원인이 요약 손실 하나뿐이라는 증명은 아니다. "
        "두 논리 단계로 구현됐지만 기존 RAG보다 우수하거나 상수 시간이라는 결론은 성립하지 않는다."
    ))
    display(measured_inputs[['provider', 'g', 'r', 'predicted_eval_input', 'observed_eval_input', 'relative_error', 'fixed_ab_stationary_group_size']].round(4))
    preparation_receipt = read_json(PARALLEL_DIR / 'preparation.json')
    prep_rows = [{'g': int(g), 'groups': v['group_count'], 'description_api_cost_usd': v['api_cost_usd_total'],
                  'original_generation_seconds': v['initial_description_seconds_total'],
                  'new_calls_in_successful_session': v['actual_new_calls_this_session']} for g,v in preparation_receipt.items()]
    display(pd.DataFrame(prep_rows).round(6))
    display(Markdown('설명 준비 비용은 완료한 설명 생성의 누적 비용이며, 중단·실패한 개발 시도의 모든 청구액을 합한 연구 총지출은 아니다. 기존 Jev 분류 준비 비용은 manifest.json의 inherited_preparation에 별도 기록돼 있다.'))


## 15. 대표 설명의 길이만 바꿔서 개발하기

**질문:** 같은 묶음과 판단 정책에서 대표 설명을 늘리면 필요한 근거를 더 잘 찾으며, 추가 비용은 얼마인가?
공통 2,000문단을 같은 100묶음으로 나눈다($g=20$). Jev가 선택하는 묶음 수 $K=5$, 최종 원문 수 $k=5$,
모델·분기/원문 판단 프롬프트·HTTP 동시성·근거 예산을 고정한다. 각 조건에서 실제 검색을 새로 실행한다.

이전 준비 프롬프트의 '짧게' 지시가 길이 증가를 막을 수 있어 **180·360·720 모두 같은 새 준비 프롬프트**로 생성한다.
다른 것은 목표 길이뿐이다. 이전 180 결과는 역사적 참고이며 이번 조건 간 비교에 섞지 않는다.
길이는 `o200k_base`로 잰 설명 JSON의 proxy이며 Jev 청구 토큰과 다르다. 목표의 80–120% 안에 들어온 비율을 확인하고
초과·미달도 실제 길이로 기록한다. 강제 잘림이나 무손실 요약을 전제하지 않는다.

각 개발 질문에서 조건 실행 순서를 무작위화한다. 이전 개발 40문항(NQ/HotpotQA 각 20)을 사용하며 B의 완전 근거 목표는 87.5%다.
이 절에서는 평가 라벨·최종 답변 생성기를 사용하지 않으므로 **답변 정확도(EM)나 전체 챗봇 비용**을 비교한 결과가 아니다.
공통 데이터·코드·모델 설정과 요청/응답을 보존하고, 실패한 시도의 비용과 완료 설명의 준비 비용을 구분한다.

In [ ]:
from rag_baselines.representation_length import run_length_study, load_length_report

RUN_LENGTH_STUDY = False  # True: 새 유료 실행. False: 완료 결과 검증·시각화만.
RUN_LENGTH_STUDY = RUN_LENGTH_STUDY or os.environ.get('DECISION_LLM_RUN_LENGTH') == '1'
LENGTH_SETTINGS = read_json(ROOT / 'config/representation_length.json')
if RUN_LENGTH_STUDY:
    from collections import deque
    from IPython.display import clear_output
    length_progress = deque(maxlen=12)
    def length_progress_update(message):
        length_progress.append(message)
        clear_output(wait=True)
        print('대표 길이 실험 · 개발 전용 · 원시 요청/응답과 부분 결과 저장 중')
        print('\n'.join(length_progress))
    LENGTH_DIR = run_length_study(ROOT, LENGTH_SETTINGS, progress=length_progress_update)
else:
    length_pointer = ROOT / 'results/current_length_run.txt'
    LENGTH_DIR = ROOT / length_pointer.read_text(encoding='utf-8').strip() if length_pointer.exists() else None
if LENGTH_DIR is not None:
    length_frame, length_summary, length_manifest, length_preparation = load_length_report(ROOT, LENGTH_DIR)
    LENGTH_SETTINGS = length_manifest['settings']
    LENGTH_ANALYSIS = LENGTH_DIR / 'analysis'
    LENGTH_ANALYSIS.mkdir(exist_ok=True)
    display(pd.DataFrame([{'항목': k, '고정/설정': str(v)} for k,v in LENGTH_SETTINGS.items()]))
    print('검증한 개발 실험:', LENGTH_DIR)
    print('원시 로그:', ROOT / length_manifest['raw_log'])
else:
    display(Markdown('**아직 대표 길이 실험의 완료 결과가 없습니다.** 15절의 실행 스위치를 켜면 준비·검색을 측정합니다.'))

## 16. 목표 숫자가 실제 설명 길이를 바꿨는지 확인하기

모든 조건에서 개체명은 최대 12개, 원문 인용 Q&A는 최대 2개다. 같은 포맷·상한·원문에서 설명 분량의 목표만 바꾼다.
공급자의 자유 생성이므로 목표 길이대로 정확히 나오지는 않는다. 아래 분포는 **정규화 후 실제로 판단 모델에 전달한 설명**의 길이다.
요약의 모든 문장이 원문에 의해 뒷받침되는지까지 자동으로 검증한 것은 아니다.

In [ ]:
if LENGTH_DIR is not None:
    descriptor_lengths = []
    for target, prepared in length_preparation.items():
        for group in read_json(ROOT / prepared['directory'] / 'groups.json'):
            descriptor_lengths.append({'descriptor_target_tokens': int(target), 'group_id': group['id'],
                'actual_tokens_proxy': group['descriptor_text_tokens_proxy']})
    descriptor_lengths = pd.DataFrame(descriptor_lengths)
    targets = sorted(length_summary.descriptor_target_tokens.astype(int))
    length_axis_labels = [f'{t}\n실제 {length_summary.set_index("descriptor_target_tokens").loc[t, "descriptor_mean_tokens_proxy"]:.0f}' for t in targets]
    fig, ax = plt.subplots(figsize=(10.2, 4.2), layout='constrained')
    ax.boxplot([descriptor_lengths.loc[descriptor_lengths.descriptor_target_tokens == t, 'actual_tokens_proxy']
                for t in targets], tick_labels=[str(t) for t in targets], showfliers=True)
    for i,t in enumerate(targets, 1):
        mean = descriptor_lengths.loc[descriptor_lengths.descriptor_target_tokens == t, 'actual_tokens_proxy'].mean()
        ax.scatter(i, mean, color=COLORS['C'], marker='D', zorder=4)
        ax.annotate(f'평균 {mean:.0f}', (i, mean), xytext=(18, 3), textcoords='offset points')
    ax.set_xlabel('설명 목표 길이 (proxy 토큰)')
    ax.set_ylabel('실제 설명 길이 (o200k_base 토큰)')
    ax.set_title('100개 묶음의 실제 대표 설명 길이 · 같은 묶음/준비 포맷')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=.2)
    fig.savefig(LENGTH_ANALYSIS / 'description_lengths.png', dpi=170)
    plt.show()
    display(length_summary[['descriptor_target_tokens', 'descriptor_mean_tokens_proxy',
        'descriptor_min_tokens_proxy', 'descriptor_max_tokens_proxy', 'descriptor_target_band_fraction']]
        .rename(columns={'descriptor_target_tokens':'목표 토큰', 'descriptor_mean_tokens_proxy':'실제 평균',
                         'descriptor_min_tokens_proxy':'최소', 'descriptor_max_tokens_proxy':'최대',
                         'descriptor_target_band_fraction':'목표 80~120% 내 비율'}).round(3))

## 17. 회수율의 증가와 감소를 질문별로 대조하기

묶음 단계가 모든 필수 근거를 포함한 사건 $H$, 최종 반환 원문이 그 근거를 모두 포함한 사건 $E$로 둔다.
원문은 선택한 묶음 안에서만 반환하므로 $E\subseteq H$다.

$$\widehat R_{\rm complete}(s)=\widehat R_{\rm macro}(s)\widehat R_{\rm micro\mid macro}(s).$$

40문항의 점 추정과 **단일 비율의 Wilson 95% 구간**을 표시한다. 조건 차이의 유의성 구간은 아니다.
같은 질문의 성공→실패와 실패→성공 건수도 함께 확인한다. 개발에 사용한 표본이므로 일반화 성능을 주장하지 않는다.

In [ ]:
if LENGTH_DIR is not None:
    def length_wilson(successes, total):
        z = 1.959963984540054
        p = successes / total
        denominator = 1 + z*z/total
        center = (p + z*z/(2*total)) / denominator
        half = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
        return center-half, center+half
    fig, ax = plt.subplots(figsize=(10.8, 5.0), layout='constrained')
    for column, label, color, offset, marker in [
        ('macro_complete', '묶음 후보의 필수 근거 전체 확보', COLORS['A'], -.09, 'o'),
        ('complete_evidence_hit', '최종 원문의 필수 근거 전체 확보', COLORS['C'], .09, 's')]:
        values, lower, upper = [], [], []
        for t in targets:
            observed = length_frame.loc[length_frame.descriptor_target_tokens == t, column]
            rate = observed.mean()
            lo,hi = length_wilson(observed.sum(), len(observed))
            values.append(rate); lower.append(rate-lo); upper.append(hi-rate)
        x = np.arange(len(targets)) + offset
        ax.errorbar(x, values, yerr=[lower,upper], marker=marker, capsize=4,
                    color=color, linestyle='-', label=label)
        for i,value in enumerate(values):
            ax.annotate(f'{value:.1%}', (x[i],value), xytext=(0, 9 if offset<0 else -20),
                        textcoords='offset points', ha='center', color=color)
    ax.axhline(length_manifest['development_target'], color=COLORS['B'], linestyle=':',
               label=f"B 개발 목표 {length_manifest['development_target']:.1%}")
    ax.set_xticks(np.arange(len(targets)), length_axis_labels)
    ax.set_ylim(0, 1.10)
    ax.set_xlabel('목표 / 실제 평균 설명 길이 (proxy 토큰)')
    ax.set_ylabel('완전 근거 확보율 / 개발 40문항')
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
    ax.set_title('설명이 길어지면 필요한 묶음을 더 잘 고르는가?')
    ax.legend(loc='lower right', frameon=False)
    ax.grid(axis='y', alpha=.2)
    fig.savefig(LENGTH_ANALYSIS / 'length_quality.png', dpi=170)
    plt.show()
    length_task_quality = length_frame.groupby(['descriptor_target_tokens','task']).agg(
        questions=('question_id','count'), macro_rate=('macro_complete','mean'),
        evidence_rate=('complete_evidence_hit','mean')).reset_index()
    display(length_task_quality.round(4))
    paired = length_frame.pivot(index='question_id', columns='descriptor_target_tokens', values='complete_evidence_hit')
    length_changes = pd.DataFrame([{'reference_target': targets[0], 'target': t, 'questions':len(paired),
        'gained_questions':int(((paired[targets[0]]==0)&(paired[t]==1)).sum()),
        'lost_questions':int(((paired[targets[0]]==1)&(paired[t]==0)).sum()),
        'delta_percentage_points':100*(paired[t]-paired[targets[0]]).mean()} for t in targets[1:]])
    display(length_changes.round(3))

## 18. 품질 제약 안에서 실제 추가 비용 비교하기

묶음 $M=\lceil N/g\rceil=100$개와 선택 원문 $Kg=100$개의 평균 판단 입력량을 $a(s),b(s)$로 둔다.

$$U(q,s)\approx M a(s)+Kg b(s),\qquad C_{\rm judge}(q,s)=p_{\rm Jev}\,U_{\rm billed}(q,s)/10^6.$$

묶음 수와 최종 판단 원문 수는 고정이지만 설명 길이와 **선택된 원문**에 따라 실제 입력량·시간이 변한다.
아래 계수는 이번 데이터의 분해이며 새로운 표본의 예측 성능을 검증한 것은 아니다.
현재 실험에서는 생성기가 없으므로 판단 API 청구액만 비교한다. 준비 비용·상속한 분류 비용은 따로 보존한다.
가능한 설정은 관측된 개발 근거 확보율이 B의 목표 이상인 설정이다. 가능한 조건이 없으면 최적값을 선언하지 않는다.

In [ ]:
if LENGTH_DIR is not None:
    from rag_baselines.jev import call_usage
    length_profiles = []
    for record in read_jsonl(LENGTH_DIR / 'retrievals.jsonl'):
        retrieval = record['retrieval']
        for stage in ['macro','micro']:
            calls = [c for c in retrieval['trace']['calls'] if c['stage'] == stage]
            usage, cost = call_usage(calls, length_manifest['c_model']['prices'])
            count = len(descriptor_lengths) // len(targets) if stage == 'macro' else len(retrieval['candidates'])
            length_profiles.append({'descriptor_target_tokens':record['descriptor_target_tokens'],
                'question_id':record['question_id'],'stage':stage,'judgments':count,
                'input_tokens':usage['input_tokens'],'input_tokens_per_judgment':usage['input_tokens']/count,
                'api_cost_usd':cost,'physical_http_requests':len(calls), 'wall_seconds':retrieval['seconds'][stage]})
    length_profiles = pd.DataFrame(length_profiles)
    length_profile_summary = length_profiles.groupby(['descriptor_target_tokens','stage']).mean(numeric_only=True).reset_index()
    display(length_profile_summary.rename(columns={
        'descriptor_target_tokens':'길이 목표', 'stage':'단계', 'judgments':'판단 수',
        'input_tokens':'입력 토큰', 'input_tokens_per_judgment':'판단당 입력',
        'api_cost_usd':'비용 USD', 'physical_http_requests':'요청 수', 'wall_seconds':'경과 초'}).round(5))
    display(length_summary[['descriptor_target_tokens','questions','complete_evidence_hit','micro_complete_given_macro',
        'decision_input_tokens','retrieval_seconds','decision_api_cost_usd','physical_http_requests','target_met']]
        .rename(columns={'descriptor_target_tokens':'길이 목표', 'questions':'문항 수',
            'complete_evidence_hit':'근거 확보율', 'micro_complete_given_macro':'묶음 확보 시 추출률',
            'decision_input_tokens':'Jev 입력 토큰', 'retrieval_seconds':'검색 초',
            'decision_api_cost_usd':'비용 USD', 'physical_http_requests':'요청 수','target_met':'목표 충족'}).round(6))
    fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.6), layout='constrained')
    for ax, field, ylabel, title, formatter in [
        (axes[0], 'decision_input_tokens', '공급자 입력 토큰 / 질문', '판단 모델이 실제 읽은 입력', lambda v:f'{v/1000:.1f}k'),
        (axes[1], 'retrieval_seconds', '초 / 질문', '실제 검색 전체 경과 시간', lambda v:f'{v:.2f}초'),
        (axes[2], 'decision_api_cost_usd', 'USD / 질문 · 생성/준비 제외', '판단 API 비용', lambda v:f'${v:.5f}')]:
        x = np.arange(len(targets))
        values = length_summary.set_index('descriptor_target_tokens').loc[targets,field].to_numpy()
        ax.plot(x, values, color=COLORS['C'], marker='o', linewidth=2)
        for i,value in enumerate(values):
            ax.annotate(formatter(value),(i,value),xytext=(0,9),textcoords='offset points',ha='center')
        ax.set_xticks(x,length_axis_labels); ax.set_xlabel('목표 / 실제 평균 길이 (proxy 토큰)')
        ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis='y',alpha=.2)
        ax.set_ylim(0, max(values)*1.22)
    axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_:f'{v/1000:.0f}k'))
    axes[2].yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:.3f}'))
    fig.suptitle('고정 g=20, K=5 · 실제 검색 40문항 × 길이 3조건 · API/감사 로그 포함')
    fig.savefig(LENGTH_ANALYSIS / 'length_cost_latency.png', dpi=170)
    plt.show()

In [ ]:
if LENGTH_DIR is not None:
    feasible = length_summary[length_summary.target_met]
    best_quality = length_summary.sort_values(['complete_evidence_hit','decision_api_cost_usd'],ascending=[False,True]).iloc[0]
    conclusion = ('**이번 개발에서 목표를 충족한 길이 조건이 없다.**' if feasible.empty else
        f"**목표를 충족한 관측 조건 중 최저 판단 API 비용은 목표 {int(feasible.sort_values('decision_api_cost_usd').iloc[0].descriptor_target_tokens)}토큰이다.**")
    display(Markdown(conclusion +
        f" 최대 관측 근거 확보율은 {best_quality.complete_evidence_hit:.1%}이며 B의 개발 목표는 {length_manifest['development_target']:.1%}다. "
        '길이별 결과는 개발 탐색이며 한 번씩 실행했다. 새로운 평가 문항의 답변 정확도나 통계적 우위를 검증한 결과로 해석하지 않는다.'))
    shortest, longest = length_summary.iloc[0], length_summary.iloc[-1]
    change = length_changes.loc[length_changes.target == longest.descriptor_target_tokens].iloc[0]
    display(Markdown(
        f"최단 목표 대비 최장 목표에서 **{int(change.gained_questions)}문항을 새로 확보하고 {int(change.lost_questions)}문항을 잃었다.** "
        f"판단 API 비용은 {longest.decision_api_cost_usd/shortest.decision_api_cost_usd-1:.1%}, "
        f"검색 시간은 {longest.retrieval_seconds/shortest.retrieval_seconds-1:.1%} 늘었다. "
        "세 목표 모두 실제 길이를 정확히 맞추지 못했으므로 목표값과 실제 분포를 함께 해석한다. "
        "관측된 품질 제약을 충족한 후보 안의 비교이며, 전체 알고리즘의 최적성을 증명한 것은 아니다."))
    length_prep_table = pd.DataFrame([{'target_tokens':int(t),'groups':p['group_count'],
        'description_api_cost_usd_completed':p['api_cost_usd_total'],
        'actual_new_calls_this_session':p['actual_new_calls_this_session'],
        'prepare_wall_seconds_this_session':p['prepare_wall_seconds_this_session']} for t,p in length_preparation.items()])
    display(length_prep_table.round(6))
    length_exports = {'description_lengths.csv':descriptor_lengths,'task_quality.csv':length_task_quality,
        'paired_quality_changes.csv':length_changes,'stage_profiles.csv':length_profiles,
        'stage_profile_summary.csv':length_profile_summary,'preparation_costs.csv':length_prep_table}
    for name,table_to_export in length_exports.items():
        table_to_export.to_csv(LENGTH_ANALYSIS / name,index=False,encoding='utf-8-sig')
    completed_length = read_json(LENGTH_DIR / 'completed.json')
    write_json(LENGTH_ANALYSIS / 'validation_checks.json', {
        'observations':len(length_frame),'questions':length_frame.question_id.nunique(),
        'development_only':length_manifest['split']=='dev' and not length_manifest['evaluation_labels_loaded'],
        'same_membership_and_fixed_policy':True,'all_original_evidence_and_metrics_validated':True,
        'question_and_summary_csv_recomputed':True,'physical_api_ledger_and_costs_reconciled':True,
        'physical_request_response_body_hashes_and_usage_charges_verified':True,
        'physical_api_attempts':completed_length['physical_api_attempts'],
        'reported_input_truncated_count':int(length_frame.input_truncated.sum()),
        'groups_and_completed_result_hashes_verified':True})
    write_json(LENGTH_ANALYSIS / 'analysis_manifest.json', {
        'scope':'Development-only length comparison. No held-out questions or answer generation.',
        'run':str(LENGTH_DIR.relative_to(ROOT)), 'source_completed_sha256':file_sha256(LENGTH_DIR / 'completed.json'),
        'notebook_source_sha256':notebook_source_sha256(ROOT / 'notebook/07_mathematical_framework.ipynb'),
        'exports_sha256':{p.name:file_sha256(p) for p in LENGTH_ANALYSIS.iterdir() if p.is_file() and p.name!='analysis_manifest.json' and p.suffix!='.html'}})
    print('검산한 표·그림:', LENGTH_ANALYSIS)